---
numbering: false
---

# 8.5. Convexity

As we saw in [Chapter 8.3](./03-gradient-descent.ipynb), gradient descent can get stuck in a local minimum that isn't a global minimum. Let's revisit the example from that section.

---

## Motivation

Use the buttons to change the initial guess, then drag the slider to step through the updates. Why do the two runs end at different points?

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BLUE, ORANGE, PINK, GREEN = '#3d81f6', '#d97706', '#d81b60', '#004d40'
PAL = 'Palatino Linotype, Palatino, serif'

def finish(fig, height=460):
    fig.update_layout(autosize=True, height=height, font=dict(family=PAL, size=14),
        paper_bgcolor='white', plot_bgcolor='white', margin=dict(l=50,r=25,t=85,b=100),
        showlegend=False)
    fig.update_xaxes(gridcolor='#eeeeee', zerolinecolor='#999999')
    fig.update_yaxes(gridcolor='#eeeeee', zerolinecolor='#999999')
    return fig

def style_2d(fig, x_range, y_range, width=None, height=450, showlegend=False):
    finish(fig,height)
    fig.update_xaxes(range=x_range); fig.update_yaxes(range=y_range)
    return fig

def slider(steps, prefix, active=0):
    return dict(active=active,currentvalue=dict(prefix=prefix),pad=dict(t=40),steps=steps)

def make_gd_explorer():
    f=lambda x:5*x**4-x**3-5*x**2+2*x-9
    df=lambda x:20*x**3-3*x**2-10*x+2
    grid=np.linspace(-1.2,1.2,400)
    fig=go.Figure(go.Scatter(x=grid,y=f(grid),line=dict(color=BLUE,width=3),name='f(x)'))
    paths=[]
    for start in [1.1,0.0]:
        path=[start]
        for k in range(80): path.append(path[-1]-.01*df(path[-1]))
        paths.append(np.array(path))
    for i,path in enumerate(paths):
        fig.add_trace(go.Scatter(x=path[:1],y=f(path[:1]),mode='markers',marker=dict(color=ORANGE,size=11),visible=(i==0),name='Current iterate'))
    steps=[]
    for k in range(81):
        steps.append(dict(label=str(k),method='restyle',args=[{'x':[[float(p[k])] for p in paths],'y':[[float(f(p[k]))] for p in paths]},[1,2]]))
    fig.update_layout(sliders=[slider(steps,'Iteration: ')],updatemenus=[dict(type='buttons',direction='right',x=0,y=1.08,xanchor='left',yanchor='bottom',buttons=[dict(label='Initial guess = 1.1',method='restyle',args=[{'visible':[True,True,False]}]),dict(label='Initial guess = 0',method='restyle',args=[{'visible':[True,False,True]}])])])
    fig.update_xaxes(title='x',range=[-1.2,1.2]);fig.update_yaxes(title='f(x)',range=[-12,-2])
    return finish(fig)

def make_secant_explorer(index):
    funcs=[lambda x:x*x,lambda x:np.abs(x),lambda x:x**3-.5*x]
    titles=['f(x) = x²','f(x) = |x|','f(x) = x³ − ½x']
    f=funcs[index]
    grid=np.linspace(-1.25,1.25,301)
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=grid,y=f(grid),line=dict(color=BLUE,width=3),hovertemplate='Input: %{x:.2f}<br>f(input): %{y:.2f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=[-1,1],y=[f(-1),f(1)],mode='lines+markers+text',text=['(-1, f(-1))','(1, f(1))'],textposition=['top right','top left'],textfont=dict(family=PAL,size=14,color=ORANGE),line=dict(color=ORANGE,width=3,dash='dash'),marker=dict(size=9)))
    if index==2:
        fig.data[1].text=['','(1, f(1))']
        fig.add_annotation(x=-1,y=f(-1),text='(-1, f(-1))',showarrow=False,xanchor='left',yanchor='top',xshift=8,yshift=-16,font=dict(family=PAL,size=14,color=ORANGE))
    fig.add_trace(go.Scatter(x=[0,0],y=[f(0),.5*(f(-1)+f(1))],mode='lines+markers',line=dict(color='#777',dash='dot'),marker=dict(color=[BLUE,ORANGE],size=10)))
    steps=[]
    for t in np.linspace(0,1,41):
        u=2*t-1
        heights=[float(f(u)),float((1-t)*f(-1)+t*f(1))]
        steps.append(dict(label=f'{t:.2f}',method='restyle',args=[{'x':[[u,u]],'y':[heights]},[2]]))
    finish(fig,440)
    fig.update_layout(title=titles[index],sliders=[slider(steps,'t = ',20)],margin=dict(l=65,r=30,t=65,b=100))
    fig.update_xaxes(range=[-1.3,1.3],title='Input')
    fig.update_yaxes(range=[-1.5,1.5] if index==2 else [-.15,1.85],title='Height')
    return fig

def make_mse_explorer():
    matrices=[np.array([[1,0],[0,1],[1,1],[2,-1.]]),np.array([[1,2],[2,4],[3,6.]])]
    targets=[np.array([1,-1,0,3.]),np.array([1,2,3.])]
    best=[np.linalg.lstsq(matrices[0],targets[0],rcond=None)[0],np.array([1.,0.])]
    v=np.array([-2.,1.]);grid=np.linspace(-2.5,3.5,90);W1,W2=np.meshgrid(grid,grid)
    fig=make_subplots(rows=1,cols=2,subplot_titles=['Independent columns','Dependent columns'],horizontal_spacing=.14)
    for j,(X,y,w) in enumerate(zip(matrices,targets,best),1):
        Z=np.mean((y[:,None,None]-X[:,0,None,None]*W1-X[:,1,None,None]*W2)**2,axis=0)
        fig.add_trace(go.Contour(x=grid,y=grid,z=Z,showscale=False,colorscale='Blues',ncontours=18),row=1,col=j)
        cs=np.linspace(-1,1,41);path=w[:,None]+v[:,None]*cs
        fig.add_trace(go.Scatter(x=path[0],y=path[1],mode='lines',line=dict(color='#555',dash='dash')),row=1,col=j)
        fig.add_trace(go.Scatter(x=[w[0]],y=[w[1]],mode='markers',marker=dict(color=ORANGE,size=12)),row=1,col=j)
    steps=[]
    for c in np.linspace(-1,1,41):
        ws=[w+c*v for w in best]
        mses=[np.mean((y-X@w)**2) for X,y,w in zip(matrices,targets,ws)]
        steps.append(dict(label=f'{c:.2f}',method='update',args=[{'x':[[w[0]] for w in ws],'y':[[w[1]] for w in ws]}, {'title.text':f'MSE: {mses[0]:.3f} (left), {mses[1]:.3f} (right)'},[2,5]]))
    fig.update_layout(title='MSE: 0.000 (left), 0.000 (right)',sliders=[slider(steps,'c = ',20)])
    fig.update_xaxes(title='w₁',range=[-2,3.5]);fig.update_yaxes(title='w₂',range=[-2.5,2])
    return finish(fig,500)

def make_hessian_explorer():
    grid=np.linspace(-2,2,71)
    X,Y=np.meshgrid(grid,grid)
    fig=make_subplots(rows=2,cols=1,specs=[[{'type':'surface'}],[{'type':'surface'}]],subplot_titles=['f(x₁, x₂) = x₁² + x₁x₂ + 2x₂²','g(x₁, x₂) = x₁² + 4x₁x₂ + x₂²'],vertical_spacing=.09)
    for row,Z in enumerate([X*X+X*Y+2*Y*Y,X*X+4*X*Y+Y*Y],1):
        fig.add_trace(go.Surface(x=X,y=Y,z=Z,colorscale='Blues',showscale=False,hovertemplate='x₁ = %{x:.2f}<br>x₂ = %{y:.2f}<br>Height = %{z:.2f}<extra></extra>'),row=row,col=1)
    finish(fig,950)
    axis=lambda title:dict(title=title,gridcolor='#eeeeee',backgroundcolor='white',zerolinecolor='#999999')
    scene=dict(xaxis=axis('x₁'),yaxis=axis('x₂'),zaxis=axis('Function value'),camera=dict(eye=dict(x=1.6,y=1.6,z=1.0)),aspectmode='cube')
    fig.update_layout(scene=scene,scene2=scene,margin=dict(l=15,r=15,t=65,b=20))
    return fig

def make_tangent_explorer():
    grid=np.linspace(-2.2,2.2,31);X,Y=np.meshgrid(grid,grid)
    f=lambda x,y:x*x+.5*x*y+y*y+1
    def plane(a):
        return f(a,-1)+(2*a-.5)*(X-a)+(.5*a-2)*(Y+1)
    fig=go.Figure([
        go.Surface(x=X,y=Y,z=f(X,Y),colorscale='Blues',showscale=False,opacity=.85),
        go.Surface(x=X,y=Y,z=plane(1),colorscale=[[0,'#efb66d'],[1,'#efb66d']],showscale=False,opacity=.65),
        go.Scatter3d(x=[1],y=[-1],z=[f(1,-1)],mode='markers',marker=dict(color=PINK,size=6))])
    steps=[]
    for a in np.linspace(-1.5,1.5,31):
        steps.append(dict(label=f'{a:.1f}',method='restyle',args=[{'x':[X,[a]],'y':[Y,[-1]],'z':[plane(a),[f(a,-1)]]},[1,2]]))
    fig.update_layout(sliders=[slider(steps,'a₁ = ',25)],scene=dict(xaxis=dict(title='x₁',range=[-2.2,2.2]),yaxis=dict(title='x₂',range=[-2.2,2.2]),zaxis=dict(title='f(x₁,x₂)',range=[-9,14]),camera=dict(eye=dict(x=1.5,y=1.5,z=1.1))))
    return finish(fig,580)

make_gd_explorer()

When we're minimizing empirical risk, a local minimum that isn't global corresponds to a parameter vector $\vec w$ that doesn't give us the smallest possible empirical risk. The gradient is $\vec 0$ there, so gradient descent stops updating the parameters.

In this section, we'll study **convex functions**, for which every local minimum is a global minimum. We'll also see what convexity tells us about the optimal parameters for least squares.

In [1]:
def make_exp_no_minimum_figure():
    x=np.linspace(-5,2,400)
    fig=go.Figure(go.Scatter(x=x,y=np.exp(x),mode='lines',line=dict(color=BLUE,width=3)))
    fig.add_hline(y=0,line_dash='dash',line_color='#999999')
    finish(fig,420)
    fig.update_layout(title='f(x) = eˣ',margin=dict(l=60,r=25,t=65,b=55))
    fig.update_xaxes(title='x',range=[-5,2]);fig.update_yaxes(title='f(x)',range=[-.3,8])
    return fig


def make_flat_bottom_figure():
    x=np.linspace(-3,3,500)
    fig=go.Figure(go.Scatter(x=x,y=np.maximum(np.abs(x)-1,0)**2,mode='lines',line=dict(color=BLUE,width=3)))
    fig.add_trace(go.Scatter(x=[-1,1],y=[0,0],mode='lines',line=dict(color=ORANGE,width=5)))
    fig.add_annotation(x=0,y=0,text='Every point in [-1, 1] is a minimizer',ax=0,ay=-85,arrowcolor='#777777',font=dict(family=PAL,size=14))
    finish(fig,420)
    fig.update_layout(margin=dict(l=60,r=25,t=40,b=55))
    fig.update_xaxes(title='x',range=[-3,3]);fig.update_yaxes(title='f(x)',range=[-.2,4.3])
    return fig


---

## Formal Definition of Convexity

You may remember from calculus that if $f''(x) \geq 0$ everywhere, then $f$ is convex. But this test requires second derivatives to exist. For instance, $f(x)=|x|$ is convex even though it isn't differentiable at $x=0$.

Also, for a vector-to-scalar function, there isn't a single second derivative to check. The corresponding test involves a matrix of second partial derivatives called the **Hessian matrix**, which we'll introduce shortly.

**We'd like a definition that works for both scalar-to-scalar and vector-to-scalar functions, whether or not they're differentiable.**

Let's start with a scalar-to-scalar function. Pick two points on its graph and draw the line segment connecting them. For a convex function, **this segment lies on or above the graph between the two points**. Recall that the line through two points on a graph is called a [secant line](../00_math_foundations/02-derivatives.ipynb#secant-lines).

In each plot below, the orange segment connects the points on the graph corresponding to the inputs $x=-1$ and $y=1$. Here, $y$ is another input, not the height of the function. Drag $t$ to compare the height of the graph (blue) with the height of the segment (orange). Which function isn't convex?

In [2]:
make_secant_explorer(0)

In [ ]:
make_secant_explorer(1)

In [ ]:
make_secant_explorer(2)

Suppose we pick two inputs $x$ and $y$, and some $t \in [0,1]$.

- The input that is a fraction $t$ of the way from $x$ to $y$ is $(1-t)x+ty$.
- The height of the secant line at that input is $(1-t)f(x)+tf(y)$.

The segment lies on or above the graph if

$$f((1-t)x+ty) \leq (1-t)f(x)+tf(y)$$

For vector-to-scalar functions, we replace the scalar inputs $x$ and $y$ with vectors $\vec x$ and $\vec y$.

:::{note} Definition: Convex Function

Suppose $f: \mathbb{R}^d \to \mathbb{R}$. We say that $f$ is **convex** if for all $\vec x,\vec y \in \mathbb{R}^d$ and all $t \in [0,1]$,

$$f((1-t)\vec x+t\vec y) \leq (1-t)f(\vec x)+tf(\vec y)$$

:::

In other words, for all inputs,

$$\begin{gathered}
\text{height of function between }(x,f(x))\text{ and }(y,f(y))\\
\leq\\
\text{height of secant line between }(x,f(x))\text{ and }(y,f(y))
\end{gathered}$$

The two heights are compared at the same input, $(1-t)x+ty$.

The inequality must hold for every pair of inputs and every $t \in [0,1]$. One segment that lies above the graph isn't enough to prove convexity, but one counterexample is enough to disprove it.

For example, consider the third plot above, $f(x)=x^3-\frac12x$. If we choose $x=-1$, $y=1$, and $t=\frac14$, the left side of the inequality is $f(-\frac12)=\frac18$, while the right side is $-\frac14$. So, $f$ isn't convex.

::::{tip} Activity 1
:class: dropdown

Use the formal definition of convexity to prove that $f(x)=|x|$ is convex. You may use the triangle inequality.

:::{seealso} Solution
:class: dropdown

For any inputs $x$ and $y$ and any $t\in[0,1]$, the triangle inequality gives

$$|(1-t)x+ty|\leq|(1-t)x|+|ty|=(1-t)|x|+t|y|$$

Here, we used the fact that both $t$ and $1-t$ are nonnegative. This is exactly the inequality in the definition of convexity, so $f(x)=|x|$ is convex.
:::
::::

<div style="position: relative; margin-bottom: 2rem; padding-bottom: 56.25%; height: 0;"><iframe title="Formal definition of convexity — Suraj Rampure" src="https://www.youtube.com/embed/u76usWZtDJw" loading="lazy" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; border: 0;"></iframe></div>


---

## Local Minimums are Global Minimums

What does this have to do with gradient descent?

**Fact**: For a convex function, every local minimum is a global minimum.

Suppose $f$ is convex, and suppose $\vec x^*$ is a local minimum. We'll prove by contradiction that $\vec x^*$ is a global minimum.

Take any other input $\vec z$. Suppose it has a smaller function value,

$$f(\vec z)<f(\vec x^*)$$

Consider the line segment from $(\vec x^*,f(\vec x^*))$ to $(\vec z,f(\vec z))$. Since $f$ is convex, the graph lies on or below this segment. For any $0<t<1$,

$$\begin{aligned}
f((1-t)\vec x^*+t\vec z)&\leq(1-t)f(\vec x^*)+tf(\vec z)\\
&<(1-t)f(\vec x^*)+tf(\vec x^*)\\
&=f(\vec x^*)
\end{aligned}$$

The strict inequality follows from our assumption that $f(\vec z)<f(\vec x^*)$ and the fact that $t>0$.

Why is this a contradiction? By choosing $t>0$ as small as we want, the input $(1-t)\vec x^*+t\vec z$ can be made arbitrarily close to $\vec x^*$. But the inequality says its function value is smaller than $f(\vec x^*)$. This contradicts the assumption that $\vec x^*$ is a local minimum.

So no input $\vec z$ can have a smaller function value, and $\vec x^*$ is a global minimum.

This doesn't mean that a convex function must have a global minimum. For instance, $f(x)=e^x$ is convex, but $e^x$ approaches $0$ as $x\to-\infty$ without ever reaching $0$.


In [3]:
make_exp_no_minimum_figure()


---

## Strict Convexity

A convex function can have more than one minimizer. For example, the function below has a minimum value of $0$, attained at every $x\in[-1,1]$.

In [4]:
make_flat_bottom_figure()


To rule out multiple minimizers, we can strengthen the definition of convexity.

:::{note} Definition: Strictly Convex Function

Suppose $f: \mathbb{R}^d \to \mathbb{R}$. We say that $f$ is **strictly convex** if for all distinct $\vec x,\vec y\in\mathbb{R}^d$ and all $t\in(0,1)$,

$$f((1-t)\vec x+t\vec y)<(1-t)f(\vec x)+tf(\vec y)$$

:::

We exclude $t=0$ and $t=1$ because the two sides are equal at the endpoints. Between the endpoints, the segment must lie strictly above the graph.

**Fact**: A strictly convex function has at most one global minimizer.

To prove this, suppose there are two different global minimizers, $\vec x^*$ and $\vec y^*$, with the same minimum value $m$. For any $t\in(0,1)$, strict convexity gives

$$f((1-t)\vec x^*+t\vec y^*)<(1-t)f(\vec x^*)+tf(\vec y^*)=(1-t)m+tm=m$$

This is a contradiction, since $m$ is the global minimum value. So, if a strictly convex function has a global minimizer, it is unique. A minimizer still doesn't have to exist, as the example $f(x)=e^x$ shows.

::::{tip} Activity 2
:class: dropdown

Is $f(x)=|x|$ strictly convex? Use $x=1$, $y=2$, and $t=\frac12$ to check the definition. Does having a unique minimizer imply that a function is strictly convex?

:::{seealso} Solution
:class: dropdown

Both sides of the inequality equal $\frac32$, so the strict inequality fails. Thus, $|x|$ is convex but not strictly convex, even though its unique minimizer is $x=0$.

Strict convexity guarantees uniqueness when a minimizer exists, but a function can have a unique minimizer without being strictly convex.
:::
::::

### Strict Convexity and Mean Squared Error

Recall that the mean squared error for a multiple linear regression model is

$$R_\text{sq}(\vec w)=\frac1n\lVert\vec y-X\vec w\rVert^2$$

When the columns of $X$ are linearly independent, there is a unique optimal parameter vector,

$$\vec w^*=(X^TX)^{-1}X^T\vec y$$

When the columns are linearly dependent, there are infinitely many optimal parameter vectors. We saw this in [Chapter 6.4](../06_linear_transformations_and_projections/04-complete-solution.ipynb).

Let's compare the mean squared error in two examples. On the left, we'll use

$$X=\begin{bmatrix}1&0\\0&1\\1&1\\2&-1\end{bmatrix},\qquad\vec y=\begin{bmatrix}1\\-1\\0\\3\end{bmatrix},\qquad\vec w^*=\begin{bmatrix}1\\-1\end{bmatrix}$$

On the right, we'll use

$$X=\begin{bmatrix}1&2\\2&4\\3&6\end{bmatrix},\qquad\vec y=\begin{bmatrix}1\\2\\3\end{bmatrix},\qquad\vec w^*=\begin{bmatrix}1\\0\end{bmatrix}$$

In each case, the given $\vec w^*$ has an MSE of $0$, because $\vec y\in\text{colsp}(X)$ and $X\vec w^*=\vec y$, so there is no error. The columns of $X$ on the left are linearly independent, while the columns of $X$ on the right are linearly dependent.

Drag $c$ to change the parameter vector in each plot according to

$$\vec w(c)=\vec w^*+c\begin{bmatrix}-2\\1\end{bmatrix}$$

The MSE changes on the left, but stays at $0$ on the right. Why?

In [5]:
make_mse_explorer()

On the right, the direction $\vec v=\begin{bmatrix}-2\\1\end{bmatrix}$ is in $\text{nullsp}(X)$, so

$$X(\vec w^*+c\vec v)=X\vec w^*+cX\vec v=X\vec w^*$$

Changing $c$ changes the parameters, but not the predictions or the mean squared error. This is a convex function with a whole line of minimizers. On the left, the mean squared error is strictly convex, and $\vec w^*$ is the unique minimizer.

---

## The Hessian matrix

Let's return to the second derivative test from calculus. For a twice continuously differentiable scalar-to-scalar function $f:\mathbb{R}\to\mathbb{R}$, $f$ is convex if and only if $f''(x)\geq0$ for every $x$.

For vector-to-scalar functions, we need to consider second partial derivatives. For example, suppose

$$f(x_1,x_2)=x_1^2+x_1x_2+2x_2^2$$

Its gradient is

$$\nabla f(\vec x)=\begin{bmatrix}2x_1+x_2\\x_1+4x_2\end{bmatrix}$$

Each component of the gradient is a scalar-valued function of $x_1$ and $x_2$. We can take the partial derivatives of each component:

$$\frac{\partial}{\partial x_1}(2x_1+x_2)=2,\qquad\frac{\partial}{\partial x_2}(2x_1+x_2)=1$$

$$\frac{\partial}{\partial x_1}(x_1+4x_2)=1,\qquad\frac{\partial}{\partial x_2}(x_1+4x_2)=4$$

We arrange these second partial derivatives in a matrix called the Hessian matrix.

:::{note} Definition: Hessian Matrix

Suppose $f:\mathbb{R}^d\to\mathbb{R}$ is twice continuously differentiable. The **Hessian matrix** of $f$ at $\vec x$ is the $d\times d$ matrix

$$H_f(\vec x)=\begin{bmatrix}
\frac{\partial^2 f}{\partial x_1^2}&\frac{\partial^2 f}{\partial x_1\partial x_2}&\cdots&\frac{\partial^2 f}{\partial x_1\partial x_d}\\
\frac{\partial^2 f}{\partial x_2\partial x_1}&\frac{\partial^2 f}{\partial x_2^2}&\cdots&\frac{\partial^2 f}{\partial x_2\partial x_d}\\
\vdots&\vdots&\ddots&\vdots\\
\frac{\partial^2 f}{\partial x_d\partial x_1}&\frac{\partial^2 f}{\partial x_d\partial x_2}&\cdots&\frac{\partial^2 f}{\partial x_d^2}
\end{bmatrix}$$

Entry $(i,j)$ is found by taking the partial derivative of the $i$th component of $\nabla f$ with respect to $x_j$.
:::

For the example above,

$$H_f(\vec x)=\begin{bmatrix}2&1\\1&4\end{bmatrix}$$

**For our purposes, the Hessian matrix is always symmetric: multivariable calculus rules tell us that $\frac{\partial^2 f}{\partial x_i\partial x_j}=\frac{\partial^2 f}{\partial x_j\partial x_i}$.** For now, we'll take this as fact. There are edge cases where the second partial derivatives aren't continuous and this equality can fail, but we won't dwell on them here.

In this case, the Hessian doesn't depend on $\vec x$. For more sophisticated functions, it *could*.

### Convexity and the Hessian

For a scalar-to-scalar function, we check that $f''(x)\geq0$. For a vector-to-scalar function, we can't just check that each entry of the Hessian is nonnegative. Instead, we need to check a quadratic form.

**Fact**: A twice continuously differentiable function $f:\mathbb{R}^d\to\mathbb{R}$ is convex if and only if

$$\vec v^T{\color{#3d81f6}H_f(\vec x)}\vec v\geq0\qquad\text{for every }\vec x\in\mathbb{R}^d\text{ and every }\vec v\in\mathbb{R}^d$$

If the inequality is strict for every $\vec x$ and every nonzero $\vec v$, then $f$ is strictly convex.

Remember that $\vec v^TA\vec v$ is called a **quadratic form**. For a fixed input $\vec x$, the condition that $\vec v^TH_f(\vec x)\vec v$ returns a nonnegative result for every possible $\vec v$ means that $H_f(\vec x)$ is **positive semidefinite**. The convexity test requires this at every input $\vec x$.

We'll take this convexity test as fact for now. We will revisit this idea in [Chapter 9.6](../09_eigenvalues_and_eigenvectors/06-rayleigh-quotient.ipynb), when we study eigenvalues and eigenvectors.


### Example: Checking Convexity

Consider the function

$$g(x_1,x_2)=x_1^2+4x_1x_2+x_2^2$$

Its Hessian is $\begin{bmatrix}2&4\\4&2\end{bmatrix}$. All of its entries are positive, but if we choose $\vec v=\begin{bmatrix}1\\-1\end{bmatrix}$, then

$$\vec v^T{\color{#3d81f6}H_g}\vec v=\begin{bmatrix}1&-1\end{bmatrix}{\color{#3d81f6}\begin{bmatrix}2&4\\4&2\end{bmatrix}}\begin{bmatrix}1\\-1\end{bmatrix}=-4$$

So, $g$ isn't convex. We can also see this by restricting $g$ to the line $x_2=-x_1$:

$$g(s,-s)=-2s^2$$

Compare this to the earlier function $f(x_1,x_2)=x_1^2+x_1x_2+2x_2^2$. The plots below show their graphs in $\mathbb{R}^3$. You can rotate each plot to examine its shape. The first is a convex bowl, while the second is a saddle.


In [6]:
make_hessian_explorer()

### Example: Mean Squared Error

In [Chapter 8.2](./02-gradients-matrix-vector-operations.ipynb), we found that

$$\nabla R_\text{sq}(\vec w)=\frac2n(X^TX\vec w-X^T\vec y)$$

How do we find the Hessian? The gradient is a vector-to-vector function, so we can't apply the gradient rules from [Chapter 8.2](./02-gradients-matrix-vector-operations.ipynb) directly to this entire expression. Instead, we'll take the partial derivatives of its components and stack those into a matrix, as we did above. You can think of this as taking the gradient of each component of $\nabla R_\text{sq}(\vec w)$ and writing its transpose as a row of the Hessian.

To simplify the notation, let $A=X^TX$ and $\vec b=X^T\vec y$. These are both constant with respect to $\vec w$, so

$$\nabla R_\text{sq}(\vec w)=\frac2n(A\vec w-\vec b)$$

If $X$ has $d$ columns, then $A$ has shape $d\times d$, and $\vec w$ and $\vec b$ each have $d$ components. Let's expand the matrix-vector product:

$$A\vec w=
\begin{bmatrix}
A_{11}&A_{12}&\cdots&A_{1d}\\
A_{21}&A_{22}&\cdots&A_{2d}\\
\vdots&\vdots&\ddots&\vdots\\
A_{d1}&A_{d2}&\cdots&A_{dd}
\end{bmatrix}
\begin{bmatrix}w_1\\w_2\\\vdots\\w_d\end{bmatrix}
=
\begin{bmatrix}
A_{11}w_1+A_{12}w_2+\cdots+A_{1d}w_d\\
A_{21}w_1+A_{22}w_2+\cdots+A_{2d}w_d\\
\vdots\\
A_{d1}w_1+A_{d2}w_2+\cdots+A_{dd}w_d
\end{bmatrix}$$

The $i$th component is the dot product of row $i$ of $A$ with $\vec w$:

$$(A\vec w)_i=A_{i1}w_1+A_{i2}w_2+\cdots+A_{id}w_d$$

To get component $i$ of the gradient, subtract component $i$ of $\vec b$ and multiply by $\frac2n$. By the definition of the gradient, this component is $\frac{\partial R_\text{sq}}{\partial w_i}$, so

$$\frac{\partial R_\text{sq}}{\partial w_i}=\frac2n\left(A_{i1}w_1+A_{i2}w_2+\cdots+A_{id}w_d-b_i\right)$$

Taking the partial derivative of this scalar expression with respect to $w_j$ leaves only the coefficient of $w_j$:

$$\frac{\partial}{\partial w_j}\left(\frac{\partial R_\text{sq}}{\partial w_i}\right)=\frac2nA_{ij}$$

This gives us entry $(i,j)$ of the Hessian. Since it holds for every $i$ and $j$, we have

$$H_{R_\text{sq}}(\vec w)=\frac2nA=\frac2nX^TX$$

Now, for any $\vec v$,

$$\vec v^TH_{R_\text{sq}}\vec v=\frac2n\vec v^TX^TX\vec v=\frac2n\lVert X\vec v\rVert^2\geq0$$

A squared norm can't be negative, so the mean squared error for a linear model is convex in $\vec w$.

If $X$ has linearly independent columns, then $X\vec v\ne\vec0$ for every nonzero $\vec v$. The squared norm is strictly positive, which makes the mean squared error strictly convex. Its minimizer exists because projecting $\vec y$ onto $\text{colsp}(X)$ gives a closest prediction vector, and independent columns give a unique parameter vector for that prediction.

If the columns are dependent, there is a nonzero $\vec v\in\text{nullsp}(X)$. Moving along this direction leaves the predictions and mean squared error unchanged, as we saw in the slider example. The mean squared error is convex, but not strictly convex.

---

## Aside: Tangent Hyperplanes

For a differentiable scalar-to-scalar function, there's another way to check convexity: the graph must lie on or above every tangent line.

For vector-to-scalar functions, the tangent line becomes a **tangent hyperplane**. When there are two input variables, this is a plane in three dimensions.

The figure below uses $f(x_1,x_2)=x_1^2+\frac12x_1x_2+x_2^2+1$. Drag the slider to change the point of tangency,

$$\vec a=\begin{bmatrix}a_1\\-1\end{bmatrix}$$

You can also rotate the plot. The orange plane touches the blue surface but never rises above it.

In [7]:
make_tangent_explorer()

Suppose we fix a point $\vec a$. The tangent hyperplane to $f$ at $\vec a$ is the linear approximation

$$
L_{\vec a}(\vec x) = f(\vec a) + \nabla f(\vec a)^T (\vec x - \vec a)
$$

This is the vector-to-scalar version of the tangent-line formula from calculus.

:::{attention} First-order characterization of convexity

If $f : \mathbb{R}^d \to \mathbb{R}$ is differentiable, then $f$ is convex if and only if

$$
f(\vec y) \ge f(\vec x) + \nabla f(\vec x)^T(\vec y - \vec x)
$$

for all $\vec x, \vec y \in \mathbb{R}^d$.

:::

The graph of a differentiable convex function lies on or above every tangent hyperplane.

This gives us another useful result. If $\nabla f(\vec x^*)=\vec0$, then the inequality becomes

$$f(\vec y)\geq f(\vec x^*)\qquad\text{for every }\vec y$$

**Fact**: For a differentiable convex function, any point with zero gradient is a global minimizer. For a general differentiable function, a zero gradient could instead indicate a local maximum or a saddle point.

That still doesn't let us choose an arbitrary step size for gradient descent. For example, on the strictly convex function $f(x)=x^2$, the update is $x^{(k+1)}=(1-2\alpha)x^{(k)}$. Starting from $x^{(0)}=1$, a step size of $\alpha=1$ makes the iterates alternate between $1$ and $-1$ forever. Convexity tells us about the function's minima, but the choice of step size still matters.
